# Plot global population map (per decade)

In [ ]:
import os
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import SymLogNorm, ListedColormap
from matplotlib.ticker import FuncFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from utils.utils import land_filter
from utils.utils import autosize_figure
import config
from utils.utils import require_dir
import pathlib

In [ ]:
# === Path config ===
POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SSP_pop" / "SSP2")
SAVE_DIR = require_dir(pathlib.Path(config.PLOTTING_ROOT) / "population")

ssp = "ssp2"

In [ ]:
def plot_population(da, SSP, year, SAVE_DIR):
    # Create figure
    fig = plt.figure(figsize=autosize_figure(1, 1))
    gs = GridSpec(2, 1, height_ratios=[15, 1])  # rows, columns

    # Map projection and display
    projection = ccrs.Robinson()
    crs = ccrs.PlateCarree()

    country_borders = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none')

    # Use log scale for normalization
    vmax = da.max().item()

    # Find positive population and zero values
    da_pos = da.where(da > 0)
    da_zero = da.where(da == 0)

    # Set colour map with zero colour
    cmap = plt.get_cmap("magma")
    zero_cmap = ListedColormap(["#bdbdbd"])

    # Define a linear threshold region around zero
    linthresh = 1

    norm = SymLogNorm(linthresh=linthresh, vmin=0, vmax=vmax, base=10)

    # First panel
    ax = fig.add_subplot(gs[0, 0], projection=projection, frameon=True)  
    cb = da_pos.plot(
        transform=crs,
        add_colorbar=False,
        cmap=cmap,
        norm=norm,
        subplot_kws={'projection': projection}
    )
    da_zero.plot(
        transform=crs,
        cmap=zero_cmap,
        add_colorbar=False,
        subplot_kws={'projection': projection}
    )

    ax.coastlines(resolution="50m", linewidth=0.75)
    ax.add_feature(country_borders, edgecolor='k', linewidth=0.75)
    plt.title(f"{SSP.upper()} Population Projections {year}", fontsize=16)

    # First colorbar
    cax = fig.add_subplot(gs[1, 0])
    col_bar = plt.colorbar(cb, cax=cax, orientation='horizontal')
    col_bar.set_label("Population", fontsize=13)

    # Ticks
    formatter = FuncFormatter(lambda v, _: f"{v:g}")
    col_bar.ax.xaxis.set_major_formatter(formatter)

    plt.tight_layout()

    out_file = f"{SSP.upper()}_population_projection_{year}.png"
    out_path = os.path.join(SAVE_DIR, out_file)
    plt.savefig(out_path)
    return

In [ ]:
# === Main loop ===
years = range(2000, 2101, 10)

for year in years:
    pop_file = f"{ssp}_coarse_grid_{year}.nc"
    pop_path = os.path.join(POP_DIR, pop_file)
    pop = xr.open_dataarray(pop_path)

    pop_land = land_filter(pop)

    plot_population(pop_land, ssp, year, SAVE_DIR)